In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style = "darkgrid")
import joblib
import shap

from src.database.connection import get_connection
from src.training.data import split_dataset, split_features
from src.training.baselines import get_baseline_predictions
from src.training.evaluation import evaluate_models

import warnings
warnings.filterwarnings(
    "ignore",
    message = "pandas only supports SQLAlchemy connectable.*",
    category = UserWarning
)
warnings.filterwarnings(
    "ignore",
    message = ".*Falling back to prediction using DMatrix due to mismatched devices.*",
    category = UserWarning
)

#### LOAD DATA AND MODELS

In [ ]:
conn = get_connection()

query = """
    SELECT 
        *
    FROM ndf;
"""

ndf = pd.read_sql(query, conn)
ndf["publish_time"] = pd.to_datetime(ndf["publish_time"], utc = True)
ndf["forecast_time"] = pd.to_datetime(ndf["forecast_time"], utc = True)

conn.close()

ndf.head()

In [ ]:
print(ndf.isna().sum(), "\n")
print(ndf.info(), "\n")
print(ndf.groupby("forecast_time").size().value_counts())

In [ ]:
data_path = "../data/dataset/modelling_dataset.parquet"
dataset = pd.read_parquet(data_path)

_, _, test_split = split_dataset(
    dataset = dataset,
    train_size = 0.7,
    validation_size = 0.15
)

X_test, y_test = split_features(test_split)

In [ ]:
ridge = joblib.load("../models/optimised_ridge.joblib")
random_forest = joblib.load("../models/optimised_random_forest.joblib")
xgboost = joblib.load("../models/optimised_xgboost.joblib")

ridge_predictions = ridge.predict(X_test)
rf_predictions = random_forest.predict(X_test)
xgb_predictions = xgboost.predict(X_test)
baseline_predictions = get_baseline_predictions(test_split)

In [ ]:
predictions = test_split[["reference_time", "target_time", "horizon", "target_demand"]].copy()

predictions["ridge"] = ridge_predictions
predictions["random_forest"] = rf_predictions
predictions["xgboost"] = xgb_predictions
predictions["baseline_30m"] = baseline_predictions["baseline_30m"]
predictions["baseline_24h"] = baseline_predictions["baseline_24h"]
predictions["baseline_7d"] = baseline_predictions["baseline_7d"]

predictions.sort_values(["reference_time", "horizon"]).head()

#### MODEL COMPARISON

In [ ]:
all_models = {
    "ridge": "Ridge",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
    "baseline_30m": "Baseline 30M",
    "baseline_24h": "Baseline 24H",
    "baseline_7d": "Baseline 7D"
}

metrics_dict = {
    "RMSE": "rmse",
    "MAE": "mae"
}

results = []
for model in all_models.keys():
    metrics = evaluate_models(
        y_test = y_test,
        predictions = predictions[model]
    )

    for key, value in metrics_dict.items():
        results.append({
            "Model": all_models.get(model),
            "Metric": key,
            "Value": metrics[value]
        })

results = pd.DataFrame(results)

plt.figure(figsize = (7, 4), constrained_layout = True)

sns.barplot(
    data = results.sort_values(["Metric", "Value"]),
    x = "Model",
    y = "Value",
    hue = "Metric"
)

plt.title("Metrics by Model")

min_y = results.min()["Value"] - 300
max_y = results.max()["Value"] + 100
plt.ylim(min_y, max_y)

plt.show()

In [ ]:
advanced_models = {
    "ridge": "Ridge",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost"
}

horizon_results = []
for horizon in range(1, 49):
    horizon_data = predictions[predictions["horizon"] == horizon]

    for model in advanced_models.keys():
        metrics = evaluate_models(
            y_test = horizon_data["target_demand"],
            predictions = horizon_data[model]
        )

        horizon_results.append({
            "Horizon": horizon,
            "Model": advanced_models.get(model),
            "MAE": metrics["mae"],
            "RMSE": metrics["rmse"]
        })

horizon_results = pd.DataFrame(horizon_results)

first_mae = horizon_results.groupby("Model")["MAE"].transform("first")
first_rmse = horizon_results.groupby("Model")["RMSE"].transform("first")
horizon_results["Normalised MAE"] = horizon_results["MAE"] / first_mae
horizon_results["Normalised RMSE"] = horizon_results["RMSE"] / first_rmse

fig, axes = plt.subplots(4, 1, figsize = (12, 12), constrained_layout = True)

for metric, ax in zip(["MAE", "Normalised MAE", "RMSE", "Normalised RMSE"], axes):
    sns.lineplot(
        data = horizon_results,
        x = "Horizon",
        y = metric,
        hue = "Model",
        ax = ax
    )

    ax.set_title(f"{metric} by Model")

plt.show()

In [ ]:
season_index = X_test.groupby("season").groups

seasons = ["Winter", "Spring", "Summer", "Autumn"]

season_results = []
for model in advanced_models.keys():
    for season in seasons:
        index = season_index.get(season)
        if index is None:
            continue

        metrics = evaluate_models(
            y_test = predictions.loc[index, "target_demand"],
            predictions = predictions.loc[index, model],
        )

        for key, value in metrics_dict.items():
            season_results.append({
                "Model": advanced_models.get(model),
                "Season": season,
                "Metric": key,
                "Value": metrics[value]
            })

season_results = pd.DataFrame(season_results)

sns.catplot(
    data = season_results,
    x = "Model",
    y = "Value",
    hue = "Metric",
    col = "Season",
    kind = "bar",
    height = 4,
    aspect = 1
)

plt.show()

In [ ]:
for model in advanced_models:
    predictions[f"{model}_error"] = (predictions[model] - predictions["target_demand"]).abs()

predictions.nlargest(20, "ridge_error")[[
    "reference_time", 
    "target_time", 
    "horizon", 
    "target_demand", 
    "ridge", 
    "ridge_error"
]]

In [ ]:
predictions.nlargest(20, "random_forest_error")[[
    "reference_time", 
    "target_time", 
    "horizon", 
    "target_demand", 
    "random_forest", 
    "random_forest_error"
]]

In [ ]:
predictions.nlargest(20, "xgboost")[[
    "reference_time", 
    "target_time", 
    "horizon", 
    "target_demand", 
    "xgboost", 
    "xgboost_error"
]]

In [ ]:
low_predictions = predictions[predictions["target_demand"] < 20000]
normal_predictions = predictions[
    (predictions["target_demand"] >= 20000) & (predictions["target_demand"] < 35000)
]
high_predictions = predictions[predictions["target_demand"] > 35000]

labels = ["demand < 20K", "20K <= demand <= 35K", "demand > 35K"]
datasets = [low_predictions, normal_predictions, high_predictions]

low_high_results = []
for model in advanced_models.keys():
    for label, dataset in zip(labels, datasets):
        metrics = evaluate_models(
            y_test = dataset["target_demand"],
            predictions = dataset[model],
            mape = True
        )

        for key, value in list(metrics_dict.items()) + [("MAPE", "mape")]:
            low_high_results.append({
                "Model": advanced_models.get(model),
                "Demand": label,
                "Metric": key,
                "Value": metrics[value]
            })

low_high_results = pd.DataFrame(low_high_results)

sns.catplot(
    data = low_high_results[low_high_results["Metric"] != "MAPE"],
    x = "Model",
    y = "Value",
    hue = "Metric",
    col = "Demand",
    kind = "bar",
    height = 4,
    aspect = 1
)

plt.show()

In [ ]:
sns.catplot(
    data = low_high_results[low_high_results["Metric"] == "MAPE"],
    x = "Model",
    y = "Value",
    hue = "Metric",
    col = "Demand",
    kind = "bar",
    height = 4,
    aspect = 1
)

plt.show()

In [ ]:
predictions["time_of_day"] = X_test["time_of_day"]

hour_results = []
for hour in np.arange(0, 24, 0.5):
    hour_data = predictions[predictions["time_of_day"] == hour]

    for model in advanced_models.keys():
        metrics = evaluate_models(
            y_test = hour_data["target_demand"],
            predictions = hour_data[model]
        )

        hour_results.append({
            "Hour": hour,
            "Model": advanced_models.get(model),
            "MAE": metrics["mae"],
            "RMSE": metrics["rmse"]
        })

hour_results = pd.DataFrame(hour_results)

fig, axes = plt.subplots(2, 1, figsize = (12, 6), constrained_layout = True)

for metric, ax in zip(["MAE", "RMSE"], axes):
    sns.lineplot(
        data = hour_results,
        x = "Hour",
        y = metric,
        hue = "Model",
        ax = ax
    )

    ax.set_title(f"{metric} by Model")

plt.show()

In [ ]:
reference_times = predictions["reference_time"].drop_duplicates()
sample = reference_times.sample(5, random_state = 123)
random_forecasts = predictions[predictions["reference_time"].isin(sample)]

fig, axes = plt.subplots(5, 1, figsize = (12, 15), constrained_layout = True)
fig.suptitle("Demand Forecasts")

for time, ax in zip(sample, axes):
    data = predictions[predictions["reference_time"] == time]

    sns.lineplot(
        data = data,
        x = "target_time",
        y = "target_demand",
        ax = ax,
        label = "Target Demand",
        color = "black",
        alpha = 0.75
    )

    for model in advanced_models:
        sns.lineplot(
            data = data,
            x = "target_time",
            y = model,
            ax = ax,
            label = model,
            linestyle = "--"
        )

    ax.set_xlabel("Time")
    ax.set_ylabel("Demand")

plt.show()

#### NDF COMPARISON

In [ ]:
ndf_compare = predictions[[
    "reference_time",
    "target_time",
    "horizon",
    "target_demand",
    "ridge",
    "random_forest",
    "xgboost"
]].copy()

ndf_compare = ndf_compare.merge(
    ndf,
    left_on = "target_time",
    right_on = "forecast_time",
    how = "left"
)

ndf_compare = ndf_compare[ndf_compare["publish_time"] <= ndf_compare["reference_time"]]

ndf_compare = ndf_compare.sort_values("publish_time").groupby(
    ["reference_time", "target_time", "horizon"], as_index = False
).tail(1).rename(columns = {
    "forecast_demand_mw": "ndf_prediction"
})

ndf_compare.head()

In [ ]:
advanced_models_with_ndf = {
    "ridge": "Ridge",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
    "ndf_prediction": "NDF Prediction"
}

ndf_results = []
for model in advanced_models_with_ndf.keys():
    metrics = evaluate_models(
        y_test = ndf_compare["target_demand"],
        predictions = ndf_compare[model]
    )

    for key, value in metrics_dict.items():
        ndf_results.append({
            "Model": advanced_models_with_ndf.get(model),
            "Metric": key,
            "Value": metrics[value]
        })

ndf_results = pd.DataFrame(ndf_results)

plt.figure(figsize = (7, 4), constrained_layout = True)

sns.barplot(
    data = ndf_results.sort_values(["Metric", "Value"]),
    x = "Model",
    y = "Value",
    hue = "Metric"
)

plt.title("Metrics by Model")

min_y = ndf_results.min()["Value"] - 300
max_y = ndf_results.max()["Value"] + 100
plt.ylim(min_y, max_y)

plt.show()

#### FEATURES - XGBOOST ONLY

In [ ]:
preprocessor = xgboost.named_steps["preprocessor"]
model = xgboost.named_steps["model"]
feature_names = preprocessor.get_feature_names_out()
feature_names = [name.split("__")[1] for name in feature_names]

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_
}).sort_values("importance", ascending = False).head(20)

sns.barplot(
    data = importance_df,
    x = "importance",
    y = "feature"
)

plt.show()

In [ ]:
X_sample = X_test.sample(n = 10000, random_state = 123)
X_sample_transformed = preprocessor.transform(X_sample)
X_sample_transformed = pd.DataFrame(
    X_sample_transformed,
    columns = feature_names,
    index = X_sample.index
)

explainer = shap.TreeExplainer(model)
shap_values = explainer(X_sample_transformed)

shap.plots.beeswarm(
    shap_values,
    max_display = 20
)